# Student Performance Analysis and Prediction System

**Minor Project**

This notebook follows the project guideline: dataset inspection, cleaning, feature engineering, descriptive statistics, EDA, required visualizations, Linear Regression, model evaluation, cross-validation, hyperparameter tuning, and interpretation.

**Dataset note:** The UCI Student Performance Mathematics dataset commonly uses `G1`, `G2`, and `G3` for the first, second, and final-period grades. It records `studytime` and `absences`; it does **not** provide a direct attendance percentage. Absences are therefore used as an attendance-related indicator, not presented as an attendance percentage.

Run the cells from top to bottom. The notebook uses the existing `src/model.py` implementation for model training and evaluation.


## 1. Import libraries and set project paths

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# The notebook is expected to be saved in the project's notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Also support opening the notebook when the current directory is the project root.
if not (PROJECT_ROOT / "data").exists() and (Path.cwd() / "notebooks").exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHART_DIR = OUTPUT_DIR / "visualizations"
RESULT_DIR = OUTPUT_DIR / "evaluation_results"

CHART_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT)
print("Data folder exists:", DATA_DIR.exists())
print("Charts will be saved to:", CHART_DIR)
print("Evaluation results will be saved to:", RESULT_DIR)


## 2. Load the student dataset

In [ ]:
# Prefer the UCI mathematics dataset filename if it exists.
candidate_files = [
    DATA_DIR / "student math.csv",
    DATA_DIR / "student_math.csv",
    DATA_DIR / "student performance dataset.csv",
]

existing_files = [path for path in candidate_files if path.exists()]

# If filenames differ, inspect CSV files and select one containing the G3 target.
if not existing_files:
    existing_files = sorted(DATA_DIR.glob("*.csv"))

if not existing_files:
    raise FileNotFoundError(
        f"No CSV dataset found in {DATA_DIR}. Place the student CSV file in the data folder."
    )

loaded = []
for path in existing_files:
    try:
        temp = pd.read_csv(path, sep=None, engine="python")
        temp.columns = temp.columns.astype(str).str.strip()
        loaded.append((path, temp))
        print(f"Found: {path.name} | shape: {temp.shape} | columns: {list(temp.columns)}")
    except Exception as exc:
        print(f"Could not read {path.name}: {exc}")

valid = [(path, frame) for path, frame in loaded if "G3" in frame.columns]
if not valid:
    raise ValueError(
        "None of the CSV files contains the required final-grade column 'G3'. "
        "Please check which file is the UCI Student Performance Mathematics dataset."
    )

# Use the first matching candidate, preferring 'student math.csv' above.
dataset_path, raw_df = valid[0]
df = raw_df.copy()

print("\nSelected dataset:", dataset_path.name)
print("Raw dataset shape:", df.shape)
display(df.head())


## 3. Inspect dataset structure and data quality

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data type"))

print("\nMissing values:")
display(df.isna().sum().to_frame("Missing values"))

print("\nDuplicate rows:", int(df.duplicated().sum()))

print("\nDataset information:")
df.info()


## 4. Clean data and prepare features

In [ ]:
# Remove exact duplicate records.
df = df.drop_duplicates().copy()

# Convert the target to numeric and remove rows without a valid final grade.
df["G3"] = pd.to_numeric(df["G3"], errors="coerce")
df = df.dropna(subset=["G3"]).copy()

# Remove impossible/non-finite target values if any are present.
df = df[np.isfinite(df["G3"])].copy()

# Feature engineering: previous-grade average, where G1 and G2 are available.
# These are previous-period grades, not the final target G3.
if {"G1", "G2"}.issubset(df.columns):
    df["previous_grade_average"] = (
        pd.to_numeric(df["G1"], errors="coerce") +
        pd.to_numeric(df["G2"], errors="coerce")
    ) / 2

# Create a clearly labelled absence indicator if the dataset has absences.
if "absences" in df.columns:
    df["absence_count"] = pd.to_numeric(df["absences"], errors="coerce")

print("Rows after duplicate removal and target cleaning:", len(df))
print("Columns after feature engineering:", df.columns.tolist())
print("\nMissing values after cleaning (features may still contain missing values):")
display(df.isna().sum().to_frame("Missing values"))


## 5. Descriptive statistics

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

descriptive_statistics = numeric_df.agg(["count", "mean", "median", "std", "min", "max"]).T
display(descriptive_statistics.round(3))

descriptive_statistics.to_csv(RESULT_DIR / "descriptive_statistics.csv")
print("Saved descriptive statistics.")


## 6. Explore academic performance relationships

In [ ]:
# Show available academic columns so interpretations are based on the actual dataset.
academic_columns = [
    column for column in
    ["studytime", "absences", "absence_count", "G1", "G2", "G3", "previous_grade_average"]
    if column in df.columns
]
print("Available academic columns:", academic_columns)

if "G3" in df.columns:
    print("\nFinal grade (G3) summary:")
    display(df["G3"].describe().to_frame("G3"))

if "studytime" in df.columns:
    print("\nFinal grade by study-time category:")
    display(df.groupby("studytime", observed=False)["G3"].agg(["count", "mean", "median"]).round(2))

if "absences" in df.columns:
    print("\nAbsences and final grade correlation:")
    print(df[["absences", "G3"]].corr().iloc[0, 1])


## 7. Required visualizations

The following cells create the required bar chart, histogram, scatter plot, box plot, and correlation heatmap. Each chart is saved in `outputs/visualizations/`.

In [ ]:
# 7.1 Bar chart: average final grade by study-time category
if "studytime" in df.columns:
    plot_data = df.groupby("studytime", observed=False)["G3"].mean()
    plt.figure(figsize=(8, 5))
    plot_data.plot(kind="bar")
    plt.title("Average Final Grade by Weekly Study Time")
    plt.xlabel("Study-time category")
    plt.ylabel("Average final grade (G3)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(CHART_DIR / "bar_average_grade_by_studytime.png", dpi=300)
    plt.show()
else:
    print("Bar chart skipped: studytime column is not available.")


In [ ]:
# 7.2 Histogram: distribution of final grades
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="G3", bins=15, kde=True)
plt.title("Distribution of Final Examination Grades")
plt.xlabel("Final grade (G3)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.savefig(CHART_DIR / "histogram_final_grades.png", dpi=300)
plt.show()


In [ ]:
# 7.3 Scatter plot: previous grade versus final grade
scatter_x = "G2" if "G2" in df.columns else (
    "studytime" if "studytime" in df.columns else None
)
if scatter_x:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=scatter_x, y="G3", alpha=0.7)
    plt.title(f"{scatter_x} and Final Examination Grade")
    plt.xlabel(scatter_x)
    plt.ylabel("Final grade (G3)")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "scatter_feature_vs_final_grade.png", dpi=300)
    plt.show()
else:
    print("Scatter plot skipped: no suitable feature column is available.")


In [ ]:
# 7.4 Box plot: final grade across study-time categories
if "studytime" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="studytime", y="G3")
    plt.title("Final Grade Distribution by Study-Time Category")
    plt.xlabel("Study-time category")
    plt.ylabel("Final grade (G3)")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "boxplot_grade_by_studytime.png", dpi=300)
    plt.show()
else:
    print("Box plot skipped: studytime column is not available.")


In [ ]:
# 7.5 Correlation heatmap: numerical academic variables
corr_columns = [
    column for column in
    ["studytime", "absences", "G1", "G2", "G3", "previous_grade_average"]
    if column in df.columns
]
if len(corr_columns) >= 2:
    plt.figure(figsize=(9, 7))
    correlation_matrix = df[corr_columns].corr(numeric_only=True)
    sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", square=True)
    plt.title("Correlation Heatmap of Academic Variables")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "correlation_heatmap.png", dpi=300)
    plt.show()
else:
    print("Heatmap skipped: not enough numerical academic columns are available.")


## 8. Linear Regression model training and evaluation

The existing `src/model.py` contains a preprocessing-and-regression pipeline. Preprocessing is fitted within the training folds, which helps prevent data leakage. The final test set is kept separate from model fitting and tuning.

In [ ]:
from src.model import train_and_evaluate_model

metrics = train_and_evaluate_model(
    data=df,
    result_dir=RESULT_DIR,
    chart_dir=CHART_DIR
)

print("\nFinal evaluation metrics:")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value}")


## 9. Review actual and predicted final grades

In [ ]:
prediction_file = RESULT_DIR / "actual_vs_predicted.csv"

if prediction_file.exists():
    actual_predicted = pd.read_csv(prediction_file)
    display(actual_predicted.head(10))
    print("\nSaved prediction comparison:", prediction_file)
else:
    print("Prediction comparison file was not created. Check the model training cell.")

saved_chart = CHART_DIR / "actual_vs_predicted.png"
print("Actual-versus-predicted chart exists:", saved_chart.exists())


## 10. Findings and interpretation

In [ ]:
print("DATA-BASED FINDINGS")
print("-------------------")

if "G2" in df.columns:
    g2_g3_corr = df[["G2", "G3"]].corr().iloc[0, 1]
    print(f"Correlation between second-period grade (G2) and final grade (G3): {g2_g3_corr:.3f}")

if "studytime" in df.columns:
    study_grade = df.groupby("studytime", observed=False)["G3"].mean()
    print("\nAverage final grade by study-time category:")
    display(study_grade.round(2).to_frame("Average G3"))

if "absences" in df.columns:
    absence_corr = df[["absences", "G3"]].corr().iloc[0, 1]
    print(f"Correlation between absences and final grade (G3): {absence_corr:.3f}")

print("\nInterpretation guidance:")
print("- Correlation describes association; it does not prove causation.")
print("- Study-time categories are coded categories, not exact hours.")
print("- Absences are an attendance-related indicator, not an attendance percentage.")
print("- Use the evaluation metrics and plots above to discuss model performance.")


## 11. Limitations and future improvements

### Limitations
- The dataset is observational and may not represent every school or student population.
- The `studytime` field is a category of weekly study time, not an exact number of hours.
- The dataset records absences rather than a direct attendance percentage.
- Relationships in the data do not establish that one factor causes a change in final grades.
- Prediction performance may change on data from other schools or academic years.

### Future improvements
- Collect a larger dataset from multiple schools and academic years.
- Include directly measured attendance percentage and study hours.
- Compare Linear Regression with additional regression models as optional extensions.
- Evaluate the model on an independent dataset from a different school or year.
- Work with educators to validate how predictions could support students.


## 12. Save final dataset overview and confirm outputs

In [ ]:
overview = {
    "dataset_file": dataset_path.name,
    "rows_after_cleaning": int(len(df)),
    "columns": int(df.shape[1]),
    "duplicate_rows_remaining": int(df.duplicated().sum()),
    "missing_values_total": int(df.isna().sum().sum()),
    "target_column": "G3",
    "features_used": [column for column in df.columns if column != "G3"],
    "note": "Dataset contains absences, not a direct attendance percentage."
}

pd.Series(overview, dtype="object").to_json(
    RESULT_DIR / "dataset_overview.json",
    indent=4
)

print("Notebook run summary")
print("--------------------")
print("Dataset:", dataset_path.name)
print("Cleaned rows:", len(df))
print("Target:", "G3")
print("Visualizations folder:", CHART_DIR)
print("Evaluation folder:", RESULT_DIR)

print("\nGenerated visualization files:")
for file in sorted(CHART_DIR.glob("*.png")):
    print("-", file.name)

print("\nGenerated evaluation files:")
for file in sorted(RESULT_DIR.iterdir()):
    if file.is_file():
        print("-", file.name)


## 13. Conclusion

This project uses student academic records to explore patterns associated with final examination grades and to build a Linear Regression model for predicting the final grade (`G3`). The analysis includes data cleaning, descriptive statistics, visual exploration, model training, test-set evaluation, cross-validation, and basic hyperparameter tuning.

The final report should quote the actual metrics generated by this notebook and explain the charts using the observed results. Do not describe correlation as causation or claim that the dataset contains exact attendance percentages when it only records absences.

**Before submission:** Run all cells from top to bottom, confirm there are no errors, save the notebook, and ensure the generated charts and evaluation files are included in the project outputs.
